# RLVR Data Attribution Experiment

**Goal:** Measure how training on Math problems (via GRPO) influences the model's zero-shot reasoning on Code problems.

**Pipeline:**
1. Load model + LoRA adapters
2. Load Math (train) and Code (test) datasets
3. Fine-tune with GRPO on Math
4. Extract **SFT gradients** for Code test prompts ($g_{test}$)
5. Extract **RLVR gradients** for Math train prompts ($g_{train}$) — simulated GRPO step
6. Build the $N_{test} \times M_{train}$ influence matrix via TracIn

## 1. Hardware Detection & Model Initialization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    ENABLE_VLLM = True
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    ENABLE_VLLM = False
else:
    DEVICE = torch.device("cpu")
    ENABLE_VLLM = False

def clear_cache():
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()

print(f"Device: {DEVICE} | vLLM enabled: {ENABLE_VLLM}")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
LEARNING_RATE = 1e-4

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
).to(DEVICE)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 2. Prepare Datasets

In [ ]:
from datasets import load_dataset

# ── Z_train: Math (GSM8K) ────────────────────────────────────────────────
math_data = load_dataset("openai/gsm8k", "main", split="train[:50]")

def format_math(example):
    return {
        "prompt": [
            {"role": "system", "content": "You are a math reasoning assistant. Think inside <think> tags, then output your answer inside <answer> tags."},
            {"role": "user", "content": example["question"]}
        ],
        "solution": example["answer"].split("#### ")[-1]
    }

train_dataset = math_data.map(format_math)

# ── Z_test: Code (MBPP) ──────────────────────────────────────────────────
code_data = load_dataset("mbpp", split="test[:5]")

def format_code(example):
    return {
        "prompt": [
            {"role": "system", "content": "You are a coding assistant."},
            {"role": "user", "content": example["text"]}
        ],
        "solution": example["code"]
    }

test_dataset = code_data.map(format_code)

print(f"Z_train (Math): {len(train_dataset)} samples")
print(f"Z_test  (Code): {len(test_dataset)} samples")

## 3. GRPO Training on Math

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from rewards import format_reward_func, accuracy_reward_func

training_args = GRPOConfig(
    output_dir="./rlvr-mac-sandbox",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    max_steps=10,
    logging_steps=2,
    bf16=True,
    use_vllm=False,
    num_generations=8,
    generation_batch_size=8,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward_func, accuracy_reward_func],
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("Starting GRPO Training...")
trainer.train()
clear_cache()

## 4. Data Attribution — Compute Gradients

**Key fix over the old pipeline:** Train gradients ($g_{train}$) are now extracted via a simulated GRPO step (generation → reward → advantage → policy loss), not SFT cross-entropy. Test gradients ($g_{test}$) remain SFT-based since we evaluate on fixed Code prompt/solution pairs.

In [ ]:
from grpo_gradients import compute_sft_gradient, compute_rlvr_gradient

# ── g_test: Code test gradients (SFT) ────────────────────────────────────
# Subset for prototyping — increase for the full experiment
N_TEST = len(test_dataset)

print(f"Computing {N_TEST} test (Code) gradients via SFT...")
test_infos = []
for idx in range(N_TEST):
    prompt = test_dataset[idx]["prompt"][1]["content"]
    solution = test_dataset[idx]["solution"]
    g = compute_sft_gradient(model, tokenizer, prompt, solution, DEVICE)
    test_infos.append({"grad": g})
    clear_cache()
    print(f"  g_test[{idx}] norm={g.norm().item():.6f}")

In [ ]:
from functools import partial

# ── g_train: Math train gradients (RLVR) ─────────────────────────────────
# Subset for prototyping — increase for the full experiment
N_TRAIN = min(10, len(train_dataset))
G = 4  # number of generations per GRPO step

# Only use format_reward_func here since accuracy_reward_func needs
# per-sample `solution` binding. To include it, wrap with functools.partial:
#   partial(accuracy_reward_func, solution=[sol] * G)
reward_funcs_for_attribution = [format_reward_func]

print(f"Computing {N_TRAIN} train (Math) gradients via simulated GRPO (G={G})...")
train_infos = []
for idx in range(N_TRAIN):
    prompt = train_dataset[idx]["prompt"][1]["content"]
    g = compute_rlvr_gradient(
        model, tokenizer, prompt, reward_funcs_for_attribution,
        G=G, device=DEVICE, enable_vllm=ENABLE_VLLM,
    )
    train_infos.append({"grad": g})
    clear_cache()
    print(f"  g_train[{idx}] norm={g.norm().item():.6f}")

## 5. Build the Influence Matrix

In [ ]:
from attribution_math import TracInInfluence, InfluenceCalculator

calculator = InfluenceCalculator(TracInInfluence(learning_rate=LEARNING_RATE))
influence_matrix = calculator.compute_matrix(test_infos, train_infos)

print(f"Influence Matrix shape: {influence_matrix.shape}")
print(f"  ({N_TEST} Code test × {N_TRAIN} Math train)\n")
print(influence_matrix)

## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(max(6, N_TRAIN * 0.8), max(3, N_TEST * 0.8)))
im = ax.imshow(influence_matrix, cmap="RdBu_r", aspect="auto",
               vmin=-np.abs(influence_matrix).max(),
               vmax=np.abs(influence_matrix).max())

ax.set_xlabel("Math Train Prompt Index")
ax.set_ylabel("Code Test Prompt Index")
ax.set_title("RLVR Influence: Math → Code (TracIn, First-Order)")
ax.set_xticks(range(N_TRAIN))
ax.set_yticks(range(N_TEST))
fig.colorbar(im, ax=ax, label="Influence Score")

for i in range(N_TEST):
    for j in range(N_TRAIN):
        ax.text(j, i, f"{influence_matrix[i, j]:.2e}",
                ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.show()

## 7. Per-Test-Prompt Analysis

In [ ]:
for i in range(N_TEST):
    row = influence_matrix[i]
    most_helpful = int(np.argmax(row))
    most_harmful = int(np.argmin(row))
    code_prompt = test_dataset[i]["prompt"][1]["content"][:80]
    print(f"Code[{i}]: \"{code_prompt}...\"")
    print(f"  Most  helpful Math train sample: index {most_helpful} (score={row[most_helpful]:.4e})")
    print(f"  Most harmful Math train sample: index {most_harmful} (score={row[most_harmful]:.4e})")
    print()